Copyright 2026 Google LLC

SPDX-License-Identifier: Apache-2.0

주의: 본 코드는 상용 배포용이 아닌 학습 및 데모용 가이드다.

용도: 구형 구글 제미나이(Google GenAI ) 모델 SDK 호출 부위를 안전하게 찾아 개발자에게 보고해 주는 정밀 스캐닝 도구다.

## 구형 제미나이 SDK 코드 스캐너 (Gemini API Old SDK Scanner )

### 1. 프로젝트 내 구형 제미나이 SDK 코드 정밀 스캔

지정한 디렉터리 경로 내의 파이썬(`.py` ) 및 쉘(`.sh` ) 스크립트 파일을 전체 탐색하여, 구형 제미나이 SDK 호출부 및 라이브러리 연동 흔적을 완벽하게 검출한다.

**상세 스캔 흐름 및 규칙:**
* **스캔 대상 탐색**: `.py` 및 `.sh` 확장자를 가진 파일들을 재귀적으로 검색하되, 숨김 디렉터리(`. ` 으로 시작하는 폴더 ) 및 이 스캐너 자신 파일은 안전하게 스킵한다.
* **검출 규칙 패턴 매칭**: 구형 SDK의 주요 컴포넌트 호출(예: `vertexai.preview.generative_models`, `genai.configure`, `import google.generativeai` 등 )을 정규 표현식으로 한 번에 필터링해 낸다.
* **감지 결과 보고**: 구형 호출부가 발견되는 즉시 정확한 행 번호와 소스 코드 라인을 실시간 출력하여 개발자의 신속한 신형 마이그레이션을 보조한다.

In [ ]:
import os
import re

SCAN_PATH = "."
FOUND_COUNT = 0

if not os.path.isdir(SCAN_PATH):
  print(f"[오류] 지정한 경로가 디렉터리가 아니거나 존재하지 않는다: {SCAN_PATH}")
else:
  print(f"[검사 시작] 구형 제미나이 SDK 코드 스캔을 시작한다...")
  print(f"대상 디렉터리: {os.path.abspath(SCAN_PATH)}")
  print("-" * 72)
  
  patterns = [
    r"from vertexai\..*generative_models",
    r"genai\.configure",
    r"import google\.generativeai",
    r"model = genai\.GenerativeModel",
    r"model\.generate_content",
    r"vertexai\.init"
  ]
  combined_pattern = re.compile("|".join(patterns))
  
  for root, dirs, files in os.walk(SCAN_PATH):
    dirs[:] = [d for d in dirs if not d.startswith('.')]
    
    for file in files:
      if not (file.endswith(".py") or file.endswith(".sh")):
        continue
        
      file_path = os.path.join(root, file)
      
      if "gemini-api-old-sdk-scanner" in file:
        continue
        
      try:
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
          lines = f.readlines()
          
        file_matches = []
        for idx, line in enumerate(lines, 1):
          if combined_pattern.search(line):
            file_matches.append((idx, line.strip()))
            
        if file_matches:
          print(f"[발견] 파일: {file_path}")
          for line_num, content in file_matches:
            print(f"  - 행 번호 {line_num}: {content}")
            FOUND_COUNT += 1
          print("-" * 72)
      except Exception as e:
        pass

  if FOUND_COUNT == 0:
    print("[성공] 구형 제미나이 SDK 사용처가 발견되지 않았다. 안전하다!")
  else:
    print(f"[완료] 총 {FOUND_COUNT}개의 구형 호출 의심 행이 발견되었다.")
    print("     (주의: 구형 SDK는 2026년 6월 24일에 종료되므로 신형 google-genai SDK로 수동 변환을 권장한다.)")